# Bronze: Raw API Ingestion
Fetches daily electricity prices (NO2) and weather observations (SN44560), logs operational metadata, and writes raw payloads to Bronze Delta tables using insert-only `MERGE`.

In [0]:
import hashlib
import json
import time
import uuid
from datetime import date, datetime, timedelta, timezone

import requests
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DateType, TimestampType, BooleanType, DoubleType
)

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

In [0]:
# Runtime parameters
dbutils.widgets.text("date_from", "2025-01-01")
dbutils.widgets.text("date_to", "2025-01-07")
dbutils.widgets.text("run_id", "")

DATE_FROM = date.fromisoformat(dbutils.widgets.get("date_from"))
DATE_TO = date.fromisoformat(dbutils.widgets.get("date_to"))
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())
FROST_CLIENT_ID = dbutils.secrets.get(
    scope="weather_price", key="frost_client_id"
)

if DATE_TO < DATE_FROM:
    raise ValueError("date_to must be on or after date_from")

In [0]:
# Schemas for Bronze and Ops DataFrames
BRONZE_SCHEMA = StructType([
    StructField("bronze_record_id", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField("request_key", StringType(), False),
    StructField("request_date", DateType(), False),
    StructField("price_area", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("source_url", StringType(), False),
    StructField("http_status", IntegerType(), False),
    StructField("payload_json", StringType(), True),
    StructField("payload_hash", StringType(), False),
    StructField("record_count", IntegerType(), False),
    StructField("error_message", StringType(), True),
    StructField("retrieved_at_utc", TimestampType(), False),
    StructField("ingestion_run_id", StringType(), False)
])

DQ_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("passed", BooleanType(), False),
    StructField("actual_value", DoubleType(), True),
    StructField("message", StringType(), True),
    StructField("checked_at_utc", TimestampType(), False)
])

In [0]:
# Helper functions for HTTP retries, row building, and Delta MERGE
def request_with_retry(url, params=None, auth=None, timeout=30,
                       max_attempts=4, backoff_seconds=2):
    last_response = None
    last_exception = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url, params=params, auth=auth, timeout=timeout,
                headers={"User-Agent": "stavanger-no2-databricks-case/1.0"}
            )
            last_response = response
            if response.status_code == 429 or response.status_code >= 500:
                if attempt < max_attempts:
                    retry_after = response.headers.get("Retry-After")
                    delay = float(retry_after) if retry_after else (
                        backoff_seconds * (2 ** (attempt - 1))
                    )
                    time.sleep(delay)
                    continue
            return response
        except requests.RequestException as exc:
            last_exception = exc
            if attempt < max_attempts:
                time.sleep(backoff_seconds * (2 ** (attempt - 1)))
                continue
    if last_response is not None:
        return last_response
    raise RuntimeError(f"HTTP request failed: {last_exception}")


def safe_record_count(payload_text):
    if not payload_text:
        return 0
    try:
        payload = json.loads(payload_text)
        if isinstance(payload, list):
            return len(payload)
        return len(payload.get("data", []))
    except (json.JSONDecodeError, TypeError):
        return 0


def canonical_payload_for_hash(payload_text):
    """Strip dynamic API metadata/timestamps before hashing to ensure idempotency."""
    try:
        parsed = json.loads(payload_text)
        # If it's a list (electricity price), strip any non-price keys if present, or sort/normalize items. For hvakosterstrommen, normalize the list of hourly objects.
        if isinstance(parsed, list):
            # Sort or clean list elements if needed, or hash the core price values
            return json.dumps(parsed, sort_keys=True)
        # If it's a dict (MET Frost), hash only the core data array, ignoring metadata like createdAt
        if isinstance(parsed, dict) and "data" in parsed:
            return json.dumps(parsed["data"], sort_keys=True)
        return json.dumps(parsed, sort_keys=True)
    except (json.JSONDecodeError, TypeError):
        return payload_text


def build_bronze_row(source_system, request_key, request_date,
                     price_area, station_id, response):
    retrieved_at = datetime.now(timezone.utc).replace(tzinfo=None)
    payload = response.text if response.status_code == 200 else None
    error_message = None if response.status_code == 200 else response.text[:2000]
    hash_input = canonical_payload_for_hash(payload) if payload is not None else (
        f"status={response.status_code}|error={error_message}"
    )
    payload_hash = hashlib.sha256(hash_input.encode("utf-8")).hexdigest()
    bronze_record_id = hashlib.sha256(
        f"{request_key}|{payload_hash}".encode("utf-8")
    ).hexdigest()
    return (
        bronze_record_id, source_system, request_key, request_date,
        price_area, station_id, response.url, response.status_code,
        payload, payload_hash, safe_record_count(payload), error_message,
        retrieved_at, RUN_ID
    )

def merge_insert_only(source_df, table_name):
    duplicate_source_keys = (
        source_df.groupBy("bronze_record_id").count()
        .filter("count > 1").count()
    )
    if duplicate_source_keys:
        raise ValueError(
            f"Source contains {duplicate_source_keys} duplicate Bronze keys"
        )
    (
        DeltaTable.forName(spark, table_name).alias("target")
        .merge(
            source_df.alias("source"),
            "target.bronze_record_id = source.bronze_record_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


def merge_dq(rows):
    dq_df = spark.createDataFrame(rows, DQ_SCHEMA)
    (
        DeltaTable.forName(
            spark, f"{CATALOG}.ops.data_quality_results"
        ).alias("target")
        .merge(
            dq_df.alias("source"),
            """
            target.run_id = source.run_id
            AND target.layer = source.layer
            AND target.check_name = source.check_name
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
# Fetch daily API payloads and MERGE into Bronze tables
price_rows = []
weather_rows = []
current_date = DATE_FROM

while current_date <= DATE_TO:
    price_url = (
        "https://www.hvakosterstrommen.no/api/v1/prices/"
        f"{current_date:%Y}/{current_date:%m-%d}_NO2.json"
    )
    price_response = request_with_retry(price_url, timeout=20)
    price_rows.append(build_bronze_row(
        "hvakosterstrommen",
        f"price|NO2|{current_date}",
        current_date, "NO2", None, price_response
    ))

    next_date = current_date + timedelta(days=1)
    weather_response = request_with_retry(
        "https://frost.met.no/observations/v0.jsonld",
        params={
            "sources": "SN44560",
            "referencetime": (
                f"{current_date}T00:00:00Z/"
                f"{next_date}T00:00:00Z"
            ),
            "elements": (
                "air_temperature,wind_speed,"
                "sum(precipitation_amount PT1H)"
            ),
            "timeresolutions": "PT1H",
            "timeoffsets": "default",
            "levels": "default",
            "qualities": "0,1,2,3,4"
        },
        auth=(FROST_CLIENT_ID, ""),
        timeout=30
    )
    weather_rows.append(build_bronze_row(
        "frost", f"weather|SN44560|{current_date}", current_date,
        "NO2", "SN44560", weather_response
    ))
    current_date = next_date

price_df = spark.createDataFrame(price_rows, BRONZE_SCHEMA)
weather_df = spark.createDataFrame(weather_rows, BRONZE_SCHEMA)

merge_insert_only(
    price_df, f"{CATALOG}.bronze.electricity_price_api_raw"
)
merge_insert_only(
    weather_df, f"{CATALOG}.bronze.weather_api_raw"
)

In [0]:
# Run Bronze data quality checks and log results
all_df = price_df.unionByName(weather_df)
failed_count = all_df.filter(
    (F.col("http_status") != 200) | (F.col("record_count") == 0)
).count()
expected_requests = all_df.count()
successful_requests = expected_requests - failed_count
checked_at = datetime.now(timezone.utc).replace(tzinfo=None)

dq_rows = [
    (RUN_ID, "bronze", "all_requests_successful", "ERROR",
     failed_count == 0, float(failed_count),
     f"{successful_requests}/{expected_requests} requests succeeded and were nonempty",
     checked_at),
    (RUN_ID, "bronze", "bronze_source_keys_unique", "ERROR",
     all_df.select("bronze_record_id").distinct().count() == expected_requests,
     float(expected_requests), "Incoming Bronze record IDs must be unique",
     checked_at)
]
merge_dq(dq_rows)

display(all_df.select(
    "source_system", "request_key", "http_status",
    "record_count", "retrieved_at_utc"
))

if failed_count > 0:
    display(all_df.filter(
        (F.col("http_status") != 200) | (F.col("record_count") == 0)
    ))
    raise RuntimeError(
        f"Bronze ingestion failed for {failed_count} request(s)"
    )

print(f"Bronze ingestion completed. run_id={RUN_ID}")

In [0]:
print("Price Bronze rows:", spark.table(f"{CATALOG}.bronze.electricity_price_api_raw").count())
print("Weather Bronze rows:", spark.table(f"{CATALOG}.bronze.weather_api_raw").count())